[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/00_simulation.ipynb)

# 00 — Generate the simulation data

**Purpose.** Run the three simulation stages end to end and explain what each
one does, so the files everything downstream reads are reproducible from this
notebook alone.

**Inputs.** `configs/simulation.yaml` — every path and every setting comes from
there. Nothing is hard-coded here.

**Outputs.**

| stage | writes |
|---|---|
| scenario | `data/external/ue_positions.csv`, `data/external/scenario.json` |
| radio map | `data/interim/radio_map.npz` |
| MDT | `data/interim/mdt.csv` |

**Before you run.** The radio map and the MDT stages need a CUDA GPU and the
Sionna-RT extra (`task sync:rt`). The radio map stage is the slow one — it traces
rays for every transmitter on every band, and how long that takes depends on the
fidelity settings in the config.

The equivalent from a shell is `task simulation`, which runs the same three
stages in the same order.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/simulation.yaml` resolve the
same way they do for `task simulation`. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not
already ship. Note that `data/` is DVC-tracked and therefore *not* part of the
clone.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = "."  # project root inside the repository

# (import name, pip name). Colab already ships numpy, pandas, matplotlib and
# seaborn, so only these are installed.
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = (checkout / SUBDIR).resolve()
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

## 1. Setup

Compose the config and seed everything. Every notebook starts the same way, so
that a cell copied between notebooks behaves identically.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()

pd.set_option("display.max_columns", 50)
cfg.simulation.output

## 2. The site layout — a one-off, not part of a run

Before any of this can run, the config needs a table of sectors: where each mast
stands, which way it points, and what tilt it starts at on each band.

That table is generated once, by `task simulation:layout`, and pasted into
`configs/simulation.yaml` under `transmitters.sectors`. The generator lays sites
out on a square lattice about the middle of the scene, then walks outward from
each ideal position until it finds a tile that is open ground with an open
surround, confirms that tile with a downward ray, and stands the mast a fixed
height above the ground it measured. A site with no such tile within reach makes
the generator fail rather than putting a mast on a roof.

It is deliberately not re-run here. The masts are surveyed against the
undisturbed city and then held fixed: perturbations model what we get wrong
about the city, not something an operator reacts to, and a real mast stays where
it was surveyed even when the survey turns out to have been wrong.

## 3. Stage 1 — the scenario

A scenario is one version of the city together with the people in it. It is
drawn once and then held fixed, so that later stages can change tilt with
nothing else moving underneath. That is what makes the KPIs a function of tilt.

What the stage does, in order:

1. **Load the scene.** The city geometry ships with Sionna-RT.
2. **Perturb the buildings.** Some are removed, others are nudged in height,
   position and rotation. This stands for survey error — the footprints a scene
   is built from are never exactly right — rather than for the city changing.
3. **Raster the scene.** A square grid is laid over it and rays are cast
   straight down inside every cell, which says how much of the cell is open
   ground and how tall whatever stands there is.
4. **Place the demand hotspots.** Their centres are drawn towards densely built
   areas, because that is both where users gather and where propagation is
   hardest. That tension is the problem the project is about.
5. **Build the time schedule.** Demand rises and falls across the day and
   carries over from one interval to the next, so load has structure rather than
   being noise a configuration cannot be tuned against.
6. **Draw the UEs.** Each interval gets a fresh crowd, drawn from the hotspots
   and from a uniform background, with every position checked to be on open
   ground and inside the region of interest.

It writes the UE table and a manifest that records what this scenario is, so it
can be regenerated and so the train/validation/test split can be made between
whole scenarios.

In [ ]:
from src.simulation import scenario

ue_file, manifest_file = scenario.generate(cfg)

## 4. Stage 2 — the radio map

The radio map is the ground truth: how strong a signal every grid cell would
receive from every transmitter.

1. **Rebuild the same city.** The scene is loaded and perturbed again from the
   same seed, so it matches the scenario exactly.
2. **Install the radio materials.** They are swapped for plain ones that do not
   recompute themselves when the carrier changes, which is what lets the low
   band be solved at all and what keeps this scenario's material draw.
3. **Add the transmitters.** Each sector is placed at the tilt the config gives
   it *for the band being solved*, so two bands on the same mast can point
   differently. That freedom is the thing being optimised.
4. **Solve.** The solver traces rays from every transmitter over the same grid
   the UEs were binned into, and the result is turned into RSRP.

One band at a time, because a scene carries a single carrier frequency. Cells no
ray reaches are left empty rather than set to zero — no coverage and no
measurement are different things.

In [ ]:
from src.simulation import radio

radio_map_file = radio.solve(cfg)

## 5. Stage 3 — the MDT

MDT is what a handset reports, as opposed to what is true.

1. **Look each UE up in the map.** Its grid cell gives the clean signal strength
   from every cell-band pair.
2. **Add measurement error.** This is receiver error only. The shadowing that
   fading models stand in for is already there, computed from the actual
   buildings, so adding fading on top would count them twice.
3. **Censor.** Part of what is left is dropped, because a real handset does not
   report every cell it can hear. The strongest cell and its close rivals are
   never dropped, so the coverage KPIs cannot be moved by censoring.

The clean map from stage 2 is written separately and is not touched here: it is
the label a surrogate learns, and noising a label would teach a model to predict
noise.

In [ ]:
from src.simulation import mdt

mdt_file = mdt.build(cfg)

## 6. What was written

These four files are the input to `01_eda.ipynb` and to everything downstream.

In [ ]:
from pathlib import Path

pd.DataFrame(
    [
        {"stage": stage, "file": str(path), "exists": Path(path).is_file()}
        for stage, path in [
            ("scenario", ue_file),
            ("scenario", manifest_file),
            ("radio map", radio_map_file),
            ("mdt", mdt_file),
        ]
    ]
)